# Train  StyleGAN2 on Women Clothes Dataset

Source:<P>

https://blog.paperspace.com/implementation-stylegan2-from-scratch/

Adapted:<P>

Antonio Esteves @ UMinho, Jan 2025<P>

---

**Table of contents**

1. [**Load all dependencies we need**](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#load-all-dependencies-we-need)
2. [Hyperparameters](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#hyperparameters)
3. [Get data loader](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#get-data-loader)
4. [Models implementation](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#models-implementation)
    1. [Noise Mapping Network](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#noise-mapping-network)
    2. [Generator](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#generator)
    3. [Generator Block](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#generator-block)
    4. [Style Block](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#style-block)
    5. [To RGB](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#to-rgb)
    6. [Convolution with Weight Modulation and Demodulation](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#convolution-with-weight-modulation-and-demodulation)
    7. [Discriminator](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#discriminator)
    8. [Discriminator Block](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#discriminator-block)
    9. [Learning-rate Equalized Linear Layer](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#learning-rate-equalized-linear-layer)
    10. [Learning-rate Equalized 2D Convolution Layer](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#learning-rate-equalized-2d-convolution-layer)
    11. [Learning-rate Equalized Weights Parameter](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#learning-rate-equalized-weights-parameter)
    12. [Perceptual path length normalization](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#perceptual-path-length-normalization)
5. [Utils](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#utils)
    1. [gradient_penalty](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#gradientpenalty)
    2. [Sample W](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#sample-w)
    3. [Generate noise](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#generate-noise)
6. [Training](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#training)
7. [Conclusion](https://blog.paperspace.com/implementation-stylegan2-from-scratch/#conclusion)

This notebook is about StyleGAN2 from the paper [Analyzing and Improving the Image Quality of StyleGAN](https://arxiv.org/pdf/1912.04958.pdf?ref=blog.paperspace.com). We will make a clean, simple, and readable implementation of StyleGAN2 using PyTorch, and try to replicate the original paper as closely as possible.

The dataset that we will use in this notebook is this [dataset](https://www.kaggle.com/datasets/tauilabdelilah/women-clothes?select=clothes&ref=blog.paperspace.com) from Kaggle, which contains 16240 upper clothes for women with 256*192 resolution.

## Load all dependencies we need

Let us start by loading all necessary dependencies.

We first import `torch` since we will use PyTorch, and from there we import `nn`. That will help us create and train the networks, and also let us import `optim`, a package that implements various optimization algorithms, such as sgd and adam. From `torchvision` we import `datasets` and `transforms` to prepare the data and apply some transforms.

We will import `functional` as `F` from `torch.nn`, `DataLoader` from `torch.utils.data` to create mini-batch sizes, `save_image` from `torchvision.utils` to save some fake samples, `log2` and `sqrt` form `math`, `numpy` for linear algebra, `os` for interaction with the operating system, `tqdm` to show progress bars, and finally `matplotlib.pyplot` to plot some images.

In [ ]:
import torch
from   torch               import nn, optim
from   torchvision         import datasets, transforms
import torch.nn.functional as     F
from   torch.utils.data    import DataLoader
from   torchvision.utils   import save_image, make_grid
from   math                import log2, sqrt
import numpy               as     np
import os
from   tqdm                import tqdm
import matplotlib.pyplot   as     plt
import wandb
import yaml
import time

%matplotlib inline

## Hyperparameters

- Specify the DATASET path.
- Specify the computing device, CUDA if GPU is available or CPU otherwise.
- Define the number of epochs as 300.
- Define a learning rate equal to 0.001.
- Set the batch size to 32.
- Define LOG_RESOLUTION as 7 because we are trying to generate 128*128 images.
- In the original paper, they set `Z_DIM` and `W_DIM` to 512, but here we set them to 256 instead to use less GPU VRAM and speed-up training. We could perhaps even get better results if we use 512.
- In StyleGAN2 we can use any loss function we want, so I opt for WGAN-GP from the paper [Improved Training of Wasserstein GANs](https://arxiv.org/pdf/1704.00028.pdf?ref=blog.paperspace.com). This loss includes the parameter $\lambda$ that is commonly set to 10, as we do here.

In [ ]:
#DATASET                 = "/home/datasets/women_clothes/" # Women clothes
#EPOCHS                  = 300
#LEARNING_RATE           = 1e-3
#BATCH_SIZE              = 32
#LOG_RESOLUTION          = 7 # for 128*128
#Z_DIM                   = 256
#W_DIM                   = 256
#LAMBDA_GP               = 10

## Configuration

In [ ]:
LOAD_TRAINED_MODEL        = False
SKIP_TRAIN_MODEL          = False

CONFIG_FILE = '../config/config_stylegan2_clothes_128x128_01.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
config['log_resolution'] = int(log2(config['image_size']))

print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

## Initializations and create necessary folders

In [ ]:
device                  = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using {device} for computing')

# Location where we will save here the images generated during WGAN-GP training
RESULTS_PATH = f'results/{config["experiment_name"]}'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location where the trained models will be saved
MODELS_PATH  = os.path.join(os.getcwd(), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(project='StyleGAN', entity='ajesteves', config=config_wandb)

## Get data loader

Let us now create a **`get_loader`** function to:

- Applies some transformation to the images (resize the images to the resolution that we want $(2^{LOG\_RESOLUTION} \times 2^{LOG\_RESOLUTION})$, convert the images to tensors, apply a Random Horizontal Flip data augmentation, and finally normalize the tensors to have the values in the range [-1,1].
- Prepares a dataset using `ImageFolder` since we have images in a folder.
- Creates a `DataLoader` that draws minibatches of shuffled images from the dataset.
- Returns the `DataLoader`.

In [ ]:
def get_loader(path, batchsize, log_resolution):
    transform = transforms.Compose(
        [
        transforms.Resize((2 ** log_resolution, 2 ** log_resolution)),
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Normalize(
            [0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5],
            ),
        ]
    )
    dataset = datasets.ImageFolder(root=path, transform=transform)
    loader  = DataLoader(
        dataset,
        batch_size=batchsize,
        shuffle=True,
    )
    return loader

## Model Implementation

Now we will implement the StyleGAN2 network with the structure documented in the paper. We will make the implementation compact but readable and understandable. The key elements of the architecture are:

- Noise mapping network.
- Weight demodulation, instead of Adaptive Instance Normalization (AdaIN).
- Skip connections, instead of progressive growing.
- Perceptual path length regularization term added to the loss.

### Noise Mapping Network

Let us create the `MappingNetwork` class, which inherits the `nn.Module`.

- The **init** method has `z_dim` and `w_dim` arguments, which define the latent z and w dimensions. This method specifies the network structure, containing 8 `EqualizedLinear` layers, a class that we will implement later and equalizes the learning rate, alternated with ReLU activation functions.
- In the **forward** method, we specify the computation graph in the forward pass. First, we apply pixel normalization to the input `x` and then we pass `x` through the mapping network.

![](../fig/stylegan2_tutorial1_map_network.png)


In [ ]:
class MappingNetwork(nn.Module):

    def __init__(self, z_dim, w_dim):
        super().__init__()
        self.mapping = nn.Sequential(
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(z_dim, w_dim)
        )

    def forward(self, x):
        x = x / torch.sqrt(torch.mean(x ** 2, dim=1, keepdim=True) + 1e-8)  # Pixel normalization
        return self.mapping(x) # Mapping Network

### Generator

The figure below illustrates the generator architecture. At the top of the figure, which corresponds to the lower resolution block, the generator receives an initial constant. Then, the structure includes a series of blocks, one per resolution. The resolution of the feature maps in block $k+1$ is twice the resolution of block $k$. The outputs from each block $k$, regular out putput and an RGB output, are both scaled up. The RGB outputs from all blocks are summed to get the final RGB image. The `toRGB` blocks also have a style modulation, which is not shown in the figure to keep it simple. 

To make the code as clean as possible, in the implementation of the generator we will use three classes that we will define later: `StyleBlock`, `toRGB`, and `GeneratorBlock`.

![](../fig/stylegan2_tutorial1_generator.png)

Figure: The generator architecture.

- The arguments of the **init** method are `log_resolution`, which is the $\log_2$ of the image resolution, `w_dim`, which is the dimensionality of $w$, `n_featurese`, which is the number of features (or channels) in the convolution layer at the highest resolution (final block), and `max_features`, which is the maximum number of features in any generator block. In this method, we calculate the number of features in each block, we set the number of generator blocks, and we initialize the trainable $4 \times 4$ initial constant with random values, we instantiate the first style block (working at $4 \times 4$ resolution), we instantiate the first `ToRGB` block, and then we instantiate the several generator blocks.
- The arguments of the **forward** method are the intermediate latent vector $w$ and input noise. $w$ feeds both "A" blocks of every generator block and its shape is $[n\_blocks, batch\_size, W\_dim]$. The input noise feeds both "B" blocks of every generator block. Each "B" blocks gets a different noise. The trainable initial constant is expanded to match the batch size and then it passes through the the first style block. The RGB output is produced. Then, we perform the computations associated with each generator block: upsampling input $x$, upsampling RGB input, apply $x$, $rgb$ and noise into the `GeneratorBlcok` to produce a new $x$ and $rgb$. Finally, the generator passes the last RGB image through a `tanh` activation function to produce the generator output. The reason for using the `tanh` is that is that we want the pixels to be in the $[-1,1]$ range.

In [ ]:
class Generator(nn.Module):

    def __init__(self, log_resolution, w_dim, n_features = 32, max_features = 256):

        super().__init__()

        features = [
            min(max_features, n_features * (2 ** i)) for i in range(log_resolution-2, -1, -1)
        ]
        self.n_blocks = len(features)

        self.initial_constant = nn.Parameter(torch.randn((1, features[0], 4, 4)))

        self.style_block = StyleBlock(w_dim, features[0], features[0])
        self.to_rgb      = ToRGB(w_dim, features[0])

        blocks = [
            GeneratorBlock(w_dim, features[i - 1], features[i]) for i in range(1, self.n_blocks)
        ]
        self.blocks = nn.ModuleList(blocks)

    def forward(self, w, input_noise):

        batch_size = w.shape[1]

        x   = self.initial_constant.expand(batch_size, -1, -1, -1)
        x   = self.style_block(x, w[0], input_noise[0][1])
        rgb = self.to_rgb(x, w[0])

        for i in range(1, self.n_blocks):
            x          = F.interpolate(x, scale_factor=2, mode="bilinear")
            x, rgb_new = self.blocks[i - 1](x, w[i], input_noise[i])
            rgb        = F.interpolate(rgb, scale_factor=2, mode="bilinear") + rgb_new

        return torch.tanh(rgb)

### Generator Block

The figure below shows the architecture of the generator block, which consists of two style blocks and a `toRGB` block. Each style block includes a convolution with $3 \times 3$ filters and the style modulation and demodulation.

![](../fig/stylegan2_tutorial1_generatorBlock.png)

Figure: Generator block architecture.

- The arguments of the **init** method are (i) `w_dim`, which is the dimensionality of the intermediate latent vector $w$, (ii) `in_features`, which is the number of input feature maps, and (iii) `out_features`, which is the number of output feature maps. This method instantiates two style blocks and a `toRGB` layer.
- The arguments of the **forward** method are (i) $x$, which are the input feature maps of the shape $[batch_size, in_features, height, width]$, (ii) latent vector $w$, with the shape $[batch_size, w_dim]$, and (iii) the input noise, which is a tuple of two noise tensors, one tensor for each style block, with shape $[batch_size, 1, height, width]$. The method performs the block forward computations, using the two style blocks and the `toRGB` block. Input $x$ passes through the first style block ans its output goes through the second style block. The output of the second style block is applied in the `toRGB` block to obtain an RGB image, and it is also the output $x$ of the generator block.

In [ ]:
class GeneratorBlock(nn.Module):

    def __init__(self, w_dim, in_features, out_features):

        super().__init__()

        self.style_block1 = StyleBlock(w_dim, in_features, out_features)
        self.style_block2 = StyleBlock(w_dim, out_features, out_features)

        self.to_rgb = ToRGB(w_dim, out_features)

    def forward(self, x, w, noise):

        x = self.style_block1(x, w, noise[0])
        x = self.style_block2(x, w, noise[1])

        rgb = self.to_rgb(x, w)

        return x, rgb

### Style Block

The figure below shows the architecture of the style block.

![](../fig/stylegan2_tutorial1_styleBlock.png)

Figure: The style block architecture.

- The arguments of the **init** method are (i) `w_dim`, which is the dimensionality of the intermediate latent vector $w$, (ii) `in_features`, which is the number of input feature maps, and (iii) `out_features`, which is the number of output feature maps. This method instantiates: (i) an `EqualizedLinear` block (block "A" in the figure), which is an equalized learning rate linear layer that we will implement later; this block obtains the style from the latent vector $w$, (ii) a `Conv2dWeightModulate` block, which is a convolution layer with weights modulated, (iii) the noise scale and bias parameters, and (iv) a `LeakyReLU` activation function.
- The arguments of the **forward** method are $x$, $w$, and input noise. This method obtains the style $s$ from latent $w$, applies the input $x$ and style $s$ to the `Conv2dWeightModulate` layer, multiplies the input noise by the noise scale parameters, adds the result of the product to the output of the convolutional layer and to the bias, and, finally, applies the activation function. This produces the output $x$ of the style block.

In [ ]:
class StyleBlock(nn.Module):

    def __init__(self, w_dim, in_features, out_features):

        super().__init__()

        # Block "A" in the figure of the architecture
        self.to_style    = EqualizedLinear(w_dim, in_features, bias=1.0)

        self.conv        = Conv2dWeightModulate(in_features, out_features, kernel_size=3)

        # Block "B" in the figure of the architecture
        self.scale_noise = nn.Parameter(torch.zeros(1))

        self.bias        = nn.Parameter(torch.zeros(out_features))

        self.activation  = nn.LeakyReLU(0.2, True)

    def forward(self, x, w, noise):

        s = self.to_style(w)
        x = self.conv(x, s)
        if noise is not None:
            x = x + self.scale_noise[None, :, None, None] * noise
        return self.activation(x + self.bias[None, :, None, None])

### toRGB Block

The figure below shows the architecture of the toRGB block.

![](../fig/stylegan2_tutorial1_toRGB.png)

Figure: The architecture of the toRGB block.

- The arguments of the **init** method are `w_dim` and `features`. The method instantiates: (i) an `EqualizedLinear` block (block "A" in the figure), which is a linear layer with weights equalized by the learning rate; this block obtains the style from the latent vector $w$, (ii) a `Conv2dWeightModulate` block, which is a convolution layer with weights modulated, (iii) the bias parameters, and (iv) a `LeakyReLU` activation function.
- The arguments of the **forward** method are $x$ and $w$. The method obtains the $style$ from latent $w$, applies the input $x$ and $style$ to the `Conv2dWeightModulate` layer, adds the output of the convolutional layer and the bias, and, finally, applies the activation function. This produces the output $x$ of the `toRGB` block.

In [ ]:
class ToRGB(nn.Module):

    def __init__(self, w_dim, features):

        super().__init__()
        self.to_style   = EqualizedLinear(w_dim, features, bias=1.0)
        self.conv       = Conv2dWeightModulate(features, 3, kernel_size=1, demodulate=False)
        self.bias       = nn.Parameter(torch.zeros(3))
        self.activation = nn.LeakyReLU(0.2, True)

    def forward(self, x, w):

        style = self.to_style(w)
        x     = self.conv(x, style)
        return self.activation(x + self.bias[None, :, None, None])


### Convolution with Weights Modulation and Demodulation

The `Conv2dWeightModulate` class implements a convolution where the weights are scaled by the style vector (modulation) and normalized (demodulation).

- The arguments of the **\_\_init\_\_** method are `in_features`, `out_features`, `kernel_size`, `demodulate`, and `eps`. `demodulate` is a boolean value to decide if we normalize the weights by their standard deviation or not, and `eps` is a small constant *ϵ* used in the normalization. The method defines the number of output features, sets demodulate, eps, and the padding size. It also creates a tensor for the Conv2d filter weights, with values normalized by the learning rate.

- The arguments of the **forward** method are the input feature maps $x$ and the style-based scaling tensor $s$. The method scales the weights of the convolution kernel (modulation), using as scaling factor the provided style $s$. Next, it normalizes the weights of the convolution kernel by dividing by their standard deviation (demodulation). Finally, it applies the convolution.

Modulation uses the next equation, where $s$ is the style, $w$ are the kernel weights, $i$ is the input channel, $j$ is the output channel, and $k$ is the kernel index.

$w'_{ijk} = s_i . w_{ijk}$

The equation for modulating (from the [research paper](https://arxiv.org/pdf/1912.04958.pdf?ref=blog.paperspace.com)).

Demodulation uses the next equation.

$w''_{ijk} = w'_{ijk} / \sqrt{\sum_{i,k} {w'_{ijk}}^2 + \epsilon}$

The equation for demodulating (from the [research paper](https://arxiv.org/pdf/1912.04958.pdf?ref=blog.paperspace.com)).

In [ ]:
class Conv2dWeightModulate(nn.Module):

    def __init__(
        self,
        in_features,
        out_features,
        kernel_size,
        demodulate = True,
        eps        = 1e-8,
        ):

        super().__init__()
        self.out_features = out_features
        self.demodulate   = demodulate
        self.padding      = (kernel_size - 1) // 2

        # Tensor weights with values normalized by the learning rate
        self.weight       = EqualizedWeight(
            [out_features, in_features, kernel_size, kernel_size]
        )
        self.eps          = eps

    def forward(self, x, s):

        b, _, h, w = x.shape

        # Weights scaling factor (is the style)
        s          = s[:, None, :, None, None]

        # LR equalized weights
        weights    = self.weight()[None, :, :, :, :]

        # Scale the weights <=> modulation
        weights    = weights * s

        # Normalize the weights dividing by the standard deviation <=> demodulation
        if self.demodulate:
            sigma_inv = torch.rsqrt((weights ** 2).sum(dim=(2, 3, 4), keepdim=True) + self.eps)
            weights   = weights * sigma_inv

        x         = x.reshape(1, -1, h, w)

        _, _, *ws = weights.shape
        weights   = weights.reshape(b * self.out_features, *ws)

        # Apply the convolution
        x         = F.conv2d(x, weights, padding=self.padding, groups=b)

        return x.reshape(-1, self.out_features, h, w)

### Discriminator

The figure below illustrates the discriminator architecture. The discriminator first transforms the input RGB image into, with the resolution equal to $2^{LOG\_RESOLUTION} \times 2^{LOG\_RESOLUTION}$, into feature maps of the same resolution but with a larger number of channels. These fetaure maps then go through a series of discriminator blocks with residual connections. In every block, the feature maps are downsampled by a factor of 2, which reduces the map size but increases the number of maps/channels.

![](../fig/stylegan2_tutorial1_discriminator.png)

Figure: The discriminator architecture.

- The arguments of the **\_\_init\_\_** method are `log_resolution`, `n_feautures`, and `max_features`. This method calculates the number of feature maps used in each discriminator block, then it instiates a layer to convert the input RGB image to `n_features` feature maps, calculates the number of discriminator blocks, instantiates the discriminator blocks, calculates the number of feature maps to use in the last convolution layer, instantiates the last convolution layer with $3 \times 3$ filters and equalized weights, and instantiates a final linear layer with equalized weights to obtain the output classification.
- The **minibatch_std** method first computes the standard deviation of each sample, across all channels and pixels, and later it computes the standard deviation per channel and concatenates the obtained values with the input image. In this way, the discriminator will get information about the variation in the batch/image.
- The **forward** method runs input $x$, of shape $[batch\_size, 3, height, width]$, run it through the `from_RGB` layer, the discriminator blocks, the minibatch standard deviation calculation, the last $3 \times 3$ convolution, the flattening operation, and final `Linear` classification layer.

In [ ]:
class Discriminator(nn.Module):

    def __init__(self, log_resolution, n_features = 64, max_features = 256):

        super().__init__()

        features = [min(max_features, n_features * (2 ** i)) for i in range(log_resolution - 1)]

        self.from_rgb = nn.Sequential(
            EqualizedConv2d(3, n_features, 1),
            nn.LeakyReLU(0.2, True),
        )
        n_blocks    = len(features) - 1
        blocks      = [DiscriminatorBlock(features[i], features[i + 1]) for i in range(n_blocks)]
        self.blocks = nn.Sequential(*blocks)

        final_features = features[-1] + 1
        self.conv      = EqualizedConv2d(final_features, final_features, 3)
        self.linear    = EqualizedLinear(2 * 2 * final_features, 1)

    def minibatch_std(self, x):
        batch_statistics = (
            torch.std(x, dim=0).mean().repeat(x.shape[0], 1, x.shape[2], x.shape[3])
        )
        return torch.cat([x, batch_statistics], dim=1)

    def forward(self, x):

        x = self.from_rgb(x)
        x = self.blocks(x)
        x = self.minibatch_std(x)
        x = self.conv(x)
        x = x.reshape(x.shape[0], -1)

        return self.linear(x)

### Discriminator Block

In the figure below we show the architecture of the discriminator block, which consists of two convolution layers, using $3 \times 3$ filters, and a downsampling operation. The block also includes a residual connection containing a downsampling operation and a $1 \times 1$ convolution.

![](../fig/stylegan2_tutorial1_discriminatorBlock.png)

Figure: The discriminator block architecture.

- The arguments of the **\_\_init\_\_** method are `in_features` and `out_features`. The method instantiates (i) a residual block, which contains a downsampling operation and a $1 \times 1$ convolution layer, and (ii) a block with two $3 \times 3$ convolutions, and a LeakyReLU activation function after each one, and a downsampling layer implemented with an Average Pooling layer. Finally, it calculates the scale factor that we will apply after the residual connection addition.
- The **forward** method passes $x$ through the residual connection and through the Conv-Conv-Downsample block. After that, it adds the ouputs of both blocks and applies the scale factor obtained by the $\_\_init\_\_$ method.

In [ ]:
class DiscriminatorBlock(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.residual = nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=2), # Downsampling using average pooling
            EqualizedConv2d(in_features, out_features, kernel_size=1)
        )

        self.block = nn.Sequential(
            EqualizedConv2d(in_features, in_features, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, True),
            EqualizedConv2d(in_features, out_features, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, True),
        )

        self.down_sample = nn.AvgPool2d(
            kernel_size=2, stride=2
        )  # Downsampling using average pooling

        self.scale = 1 / sqrt(2)

    def forward(self, x):
        residual = self.residual(x)

        x = self.block(x)
        x = self.down_sample(x)

        return (x + residual) * self.scale

### Learning-rate Equalized Linear Layer

Here we implement the **EqualizedLinear** class, which is a `Linear` layer with the weights equalized by the learning rate.

- The arguments of the **\_\_init\_\_** method are `in_features`, `out_features`, and `bias`. The method creates a tensor for the weights, with equalized values, and creates the bias tensor.
- The **forward** method passes input $x$ through a `Linear` layer with the weights and bias defined by the `__init__` method.

In [ ]:
class EqualizedLinear(nn.Module):

    def __init__(self, in_features, out_features, bias = 0.):

        super().__init__()
        self.weight = EqualizedWeight([out_features, in_features])
        self.bias   = nn.Parameter(torch.ones(out_features) * bias)

    def forward(self, x: torch.Tensor):
        return F.linear(x, self.weight(), bias=self.bias)

### Learning-rate Equalized 2D Convolution Layer

Now we implement the **EqualizedConv2d** class, which is a 2D convolutional layer with the kernel weights equalized by the learning rate.

- The arguments of the **\_\_init\_\_** method are `in_features`, `out_features`, `kernel_size`, and `padding`. The method defines the Conv2d padding, creates a tensor for the weights with equalized values, and creates the bias tensor.
- The **forward** method passes input $x$ through a convolution layer with kernel weights, bias, and padding defined by the `__init__` method.

In [ ]:
class EqualizedConv2d(nn.Module):

    def __init__(
        self,
        in_features,
        out_features,
        kernel_size,
        padding = 0,
        ):

        super().__init__()
        self.padding = padding
        self.weight  = EqualizedWeight([out_features, in_features, kernel_size, kernel_size])
        self.bias    = nn.Parameter(torch.ones(out_features))

    def forward(self, x: torch.Tensor):
        return F.conv2d(x, self.weight(), bias=self.bias, padding=self.padding)

### Learning-rate Equalized Weights Parameter

Here, we implement the **EqualizedWeight** class that is used in the Learning-rate Equalized Linear Layer and Learning-rate Equalized 2D Convolution Layer.

This is based on the equalized learning rate technique introduced in ProGAN. According to this technique, instead of initializing the network weights with values from $\mathcal{N}(0,c)$, we initialize weights with values from $\mathcal{N}(0,1)$ and then we multiply the weghts by $c$ when they are used.

- Given the shape of the weights parameter, the **\_\_init\_\_** method initializes (i) the constant $c$ with $1/\sqrt{shape[1:]}$ and (ii) the weights with values from $\mathcal{N}(0,1)$.
- The **forward** method scales the weights by $c$.

In [ ]:
class EqualizedWeight(nn.Module):

    def __init__(self, shape):

        super().__init__()

        self.c      = 1 / sqrt(np.prod(shape[1:]))
        self.weight = nn.Parameter(torch.randn(shape))

    def forward(self):
        return self.weight * self.c

### Perceptual path length normalization

Perceptual path length regularization encourages a fixed-size step in $\mathbf{w}$ to result in a fixed-magnitude change in the generated image. The regularization term is expressed by the next equation.

$\mathbb{E}_{\mathbf{w},\mathbf{y} \sim \mathcal{N}(0, \mathbf{I})} \left( \parallel \mathbf{J}_\mathbf{w}^T \mathbf{y} \parallel_2 - a \right)^2$

Equation from the [research paper](https://arxiv.org/pdf/1912.04958.pdf?ref=blog.paperspace.com)

Where $\mathbf{J}_\mathbf{w}$ is the Jacobian matrix and is calculated with the equation below, $\mathbf{w}$ is sampled from the mapping network, $g(\mathbf{w}): \mathbf{w} \mapsto \mathbf{y}$ represents the generator, $\mathbf{y}$ are random images with pixel intensitiesfollowing the normal distribution $\mathcal{N}(0, \mathbf{I})$, and $a$ is the exponential moving average of the lengths $\parallel \mathbf{J}_\mathbf{w}^T \mathbf{y} \parallel_2$.

$\mathbf{J}_\mathbf{w} = \partial g(\mathbf{w}) / \partial \mathbf{w}$

Equation from the [research paper](https://arxiv.org/pdf/1912.04958.pdf?ref=blog.paperspace.com)

- Given the constant $\beta$ used to calculate $a$ with the exponential moving average, the **\_\_init\_\_** method initializes `beta`, the number of steps used in the calculations ($N$), and the exponential sum of $J_w^T y$.
- The arguments of the **forward** method are the latents $w$ with a shape of $[batch\_size, W\_DIM]$ and $x$, the generated images of shape $[batch\_size, 3, height, width]$. The method gets the computation device, calculates the number of pixels in the image, calculates the equations above, updates the exponential sum, increments $N$, and returns the regularization term that will be added to the loss.

In [ ]:
class PathLengthPenalty(nn.Module):

    def __init__(self, beta):

        super().__init__()

        self.beta      = beta
        self.steps     = nn.Parameter(torch.tensor(0.), requires_grad=False)
        self.exp_sum_a = nn.Parameter(torch.tensor(0.), requires_grad=False)

    def forward(self, w, x):

        device     = x.device

        # Number of pixels in the image
        image_size = x.shape[2] * x.shape[3]

        # Random image with normally distributed pixel intensities
        y          = torch.randn(x.shape, device=device)

        output     = (x * y).sum() / sqrt(image_size)
        sqrt(image_size)

        gradients, *_ = torch.autograd.grad(
            outputs      = output,
            inputs       = w,
            grad_outputs = torch.ones(output.shape, device=device),
            create_graph = True,
        )

        norm      = (gradients ** 2).sum(dim=2).mean(dim=1).sqrt()

        if self.steps > 0:
            a    = self.exp_sum_a / (1 - self.beta ** self.steps)
            loss = torch.mean((norm - a) ** 2)
        else:
            loss = norm.new_tensor(0)

        mean = norm.mean().detach()
        self.exp_sum_a.mul_(self.beta).add_(mean, alpha=1 - self.beta)
        self.steps.add_(1.)

        return loss

## Utilities

### Calculate the WGAN-GP loss

The `gradient_penalty` function calculates the WGAN-GP loss.

In [ ]:
def gradient_penalty(discriminator, real, fake, device="cpu"):

    BATCH_SIZE, C, H, W = real.shape
    beta = torch.rand((BATCH_SIZE, 1, 1, 1)).repeat(1, C, H, W).to(device)
    interpolated_images = real * beta + fake.detach() * (1 - beta)
    interpolated_images.requires_grad_(True)

    # Calculate discriminator scores
    mixed_scores = discriminator(interpolated_images)

    # Take the gradient of the scores with respect to the images
    gradient = torch.autograd.grad(
        inputs       = interpolated_images,
        outputs      = mixed_scores,
        grad_outputs = torch.ones_like(mixed_scores),
        create_graph = True,
        retain_graph = True,
    )[0]
    gradient         = gradient.view(gradient.shape[0], -1)
    gradient_norm    = gradient.norm(2, dim=1)
    gradient_penalty = torch.mean((gradient_norm - 1) ** 2)

    return gradient_penalty

### Draw a minibatch of samples for **w**

This function draws a minibatch of samples for $\mathbf{z}$, from the normal distribution, and maps these samples to the intermediate latent space using the mapping network, to get a minibatch of samples for $\mathbf{w}$.

In [ ]:
def get_w(batch_size, log_resolution, z_dim, device):

    z = torch.randn(batch_size, z_dim).to(device)
    w = mapping_network(z)
    return w[None, :, :].expand(log_resolution, -1, -1)

### Generate noise

The `get_noise` function generates two minibatches of noise to use in a generator block.

In [ ]:
def get_noise(batch_size, log_resolution):

        noise      = []
        resolution = 4

        for i in range(log_resolution):
            if i == 0:
                n1 = None
            else:
                n1 = torch.randn(batch_size, 1, resolution, resolution, device=device)
            n2 = torch.randn(batch_size, 1, resolution, resolution, device=device)

            noise.append((n1, n2))

            resolution *= 2

        return noise

### Generate new images

The `generate_examples` function uses the generator `generator`, the log2 of the resolution, the dimensionality `w_dim` of the intermediate latent vectors, the `epoch` number, and a number `n`=100, to generate `n` images and uses the `epoch` to identify the folder where the images are saved.

In [ ]:
def generate_examples(generator, log_resolution, z_dim, epoch, n=100, device=device):

    generator.eval()
    alpha = 1.0
    for i in range(n):
        with torch.no_grad():
            w     = get_w(1, log_resolution, z_dim, device)
            noise = get_noise(1, log_resolution)
            img   = generator(w, noise)
            if not os.path.exists(f'saved_examples/epoch{epoch}'):
                os.makedirs(f'saved_examples/epoch{epoch}')
            save_image(img*0.5+0.5, f"saved_examples/epoch{epoch}/img_{i}.png")

    generator.train()

The `generate_grid_images` function uses the generator `generator`, to generate a grid of `grid_W_H` by `grid_W_H` images. The function shows the grid of images and saves the grid to a file.

In [ ]:
def generate_grid_images(generator, num_grids, grid_W_H, epoch, config, device):

    grid_size = grid_W_H ** 2
    assert config["batch_size"] >= grid_size, f'Grid size must be less or equal to batch size={config["batch_size"]}'

    generator.eval()

    with torch.inference_mode():

        for num in range(num_grids):

            w     = get_w(config['batch_size'], config['log_resolution'], config['z_dim'], device)
            noise = get_noise(config['batch_size'], config['log_resolution'])
            fake  = generator(w, noise).detach().cpu()

            if(config['batch_size'] > grid_size):
                fake = fake[:grid_size]

            # Create a grid with the generated images
            grid = make_grid(fake, nrow=grid_W_H, padding=2, normalize=True)
            grid = grid.permute(1, 2, 0)
            grid = grid.numpy()

            # Display the grid of images
            _ = plt.figure(figsize=(10, 10), constrained_layout=True)
            plt.imshow(grid)

            # Save the grid of images as a PNG file
            file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_epoch{str(epoch).zfill(3)}_{str(num+1).zfill(3)}.png'
            plt.imsave(file_png, grid)

### Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(
        mapping_net,
        generator,
        discriminator,
        optimizer_m,
        optimizer_g,
        optimizer_d,
        results,
        epoch,
        hyperparameters,
        file_name,
    ):
    '''
    - saves the mapping network
    - saves the generator network
    - saves the discriminator network
    - saves the mapping network optimizer state
    - saves the generator optimizer state
    - saves the discriminator optimizer state
    - saves the average results obtained
    - saves the current epoch ID
    - saves the training hyperparameters
    '''
    results_to_save = {
        'mapping_net':     mapping_net.state_dict(),
        'generator':       generator.state_dict(),
        'discriminator':   discriminator.state_dict(),
        'm_optimizer':     optimizer_m.state_dict(),
        'g_optimizer':     optimizer_g.state_dict(),
        'd_optimizer':     optimizer_d.state_dict(),
        'results':         results,
        'epoch':           epoch,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(
        mapping_net,
        discriminator,
        generator,
        optimizer_m,
        optimizer_g,
        optimizer_d,
        file_name,
        device,
    ):
    '''
    Given instances of the mapping network, generator and discriminator models, loads from file 'file_name':
    (i)   the weights all three models,
    (ii)  the three optimizers state,
    (iii) the results obtained during model training and
    (iv)  the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    mapping_net.load_state_dict(results_loaded['mapping_net'])
    mapping_net.to(device)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    optimizer_m.load_state_dict(results_loaded['m_optimizer'])
    optimizer_g.load_state_dict(results_loaded['g_optimizer'])
    optimizer_d.load_state_dict(results_loaded['d_optimizer'])

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['epoch'], results_loaded['hyperparameters']

### Format a given time in seconds as DD HH mm ss

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

## Train the StyleGAN2

Here, we will train the StyleGAN2 model.

So, we create a function that implements the traing loop. The arguments of this function are the discriminator `discriminator`, the generator `generator`, the perceptual path length regularization term of the loss `path_length_penalty` that we will use every 16 epochs, that DataLoader `loader`, and the optimizers for the networks, `opt_generator` and `opt_mapping_network`. The main loop iterates over all the minibatches prrovided by the DataLoader, and we just use the images because we do not need the labels.

Then we optimize the discriminator (or critic), where we want to maximize $E(discriminator(real)) - E(discriminator(fake))$. This equation means how much the discriminator can distinguish between real and fake images.

After that, we optimize the mapping network and the generator, where we want to maximize $E(discriminator(fake))$, and we add to the loss function the perceptual path length term every 16 epochs.

In [ ]:
def train_step(
    mapping_net,   # mapping network
    generator,     # generator network
    discriminator, # discriminator network
    path_length_penalty,
    loader,
    opt_mapping_network,
    opt_generator,
    opt_discriminator,
    epoch,
    config,
    device,
    ):

    plp_log_interval = int(config["log_interval"] / 16)

    loop = tqdm(loader, leave=True)

    for batch_idx, (real, _) in enumerate(loop):
        real           = real.to(device)
        cur_batch_size = real.shape[0]

        w     = get_w(cur_batch_size, config['log_resolution'], config['z_dim'], device)
        noise = get_noise(cur_batch_size, config['log_resolution'])
        with torch.amp.autocast('cuda'):
            fake               = generator(w, noise)
            discriminator_fake = discriminator(fake.detach())

            discriminator_real = discriminator(real)
            gp = gradient_penalty(discriminator, real, fake, device=device)
            loss_discriminator = (
                -(torch.mean(discriminator_real) - torch.mean(discriminator_fake))
                + config['lambda_gp'] * gp
                + (0.001 * torch.mean(discriminator_real ** 2))
            )

        # optimize the discriminator
        discriminator.zero_grad()
        loss_discriminator.backward()
        opt_discriminator.step()

        generator_fake = discriminator(fake)
        loss_generator = -torch.mean(generator_fake)

        if batch_idx % 16 == 0:
            plp = path_length_penalty(w, fake)
            if not torch.isnan(plp):
                loss_generator = loss_generator + plp

        # optimize the mapping network and the generator
        mapping_net.zero_grad()
        generator.zero_grad()
        loss_generator.backward()
        opt_generator.step()
        opt_mapping_network.step()

        # Print progress metrics ...............................................
        loop.set_postfix(
            epoch = epoch+1,
            GP    = gp.item(),
            lossD = loss_discriminator.item(),
            lossG = loss_generator.item(),
            PLP   = plp.item(),
        )

        # ------------------------------------
        # Save the results in a dictionary
        # ------------------------------------

        results["g_loss"].append(loss_generator.item())
        results["d_loss"].append(loss_discriminator.item())
        results["gp"].append(gp.item())
        if batch_idx % 16 == 0:
            results["plp"].append(plp.item())

        # Save progress metrics to W&B ..........................................
        if (batch_idx+1) % config["log_interval"] == 0:

            mean_g_loss      = np.mean(results["g_loss"][-config["log_interval"]:])
            mean_d_loss      = np.mean(results["d_loss"][-config["log_interval"]:])
            mean_gp          = np.mean(results["gp"][-config["log_interval"]:])
            mean_plp         = np.mean(results["plp"][-plp_log_interval:])

            try:
                # Log metrics to Weights & Biases ............................
                wandb.log(
                    {
                    "g_loss":   mean_g_loss,
                    "d_loss":   mean_d_loss,
                    "gp":       mean_gp,
                    "plp":      mean_plp,
                    "epoch":    epoch+1,
                    }
                )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')


First, we initialize the DataLoader, instantiate the networks, select the optimizers, and put the networks in the training mode.

In [ ]:
# Create the DataLoader
loader              = get_loader(config["dataset_path"], config['batch_size'], config['log_resolution'])

# Instantiate the networks
mapping_network     = MappingNetwork(config['z_dim'], config['w_dim']).to(device)
generator           = Generator(config['log_resolution'], config['w_dim']).to(device)
discriminator       = Discriminator(config['log_resolution']).to(device)
path_length_penalty = PathLengthPenalty(0.99).to(device)

# Select the optimizers
opt_mapping_network = optim.Adam(
    mapping_network.parameters(), 
    lr    = config['lr_m'], 
    betas = (config['beta1'], config['beta2']),
)
opt_generator       = optim.Adam(
    generator.parameters(),
    lr    = config['lr_g'],
    betas = (config['beta1'], config['beta2']),
)
opt_discriminator   = optim.Adam(
    discriminator.parameters(),
    lr    = config['lr_d'],
    betas = (config['beta1'], config['beta2']),
)

# Put the networks in training mode
mapping_network.train()
generator.train()
discriminator.train()

Now let us train the networks using the training loop, and save some generated samples every 50 epoch.

In [ ]:
# Create an empty dictionary to store the training results .................
results = {
    'g_loss':              [],
    'd_loss':              [],
    'gp':                  [],
    'plp':                 [],
    'epoch_training_time': [],
}

loader = get_loader(config["dataset_path"], config['batch_size'], config['log_resolution'])

for epoch in range(config['epochs']):

    ts  = time.time()

    train_step(
        mapping_network,
        generator,
        discriminator,
        path_length_penalty,
        loader,
        opt_mapping_network,
        opt_generator,
        opt_discriminator,
        epoch,
        config,
        device,
    )

    te        = time.time()
    texec_sec = te - ts
    texec_str = time_format(texec_sec)
    print(f'Epoch training time: {texec_str}')
    results['epoch_training_time'].append(texec_sec)

    try:
        wandb.log(
            {
            "epoch_training_time_sec": texec_sec,
            }
        )
    except Exception as ex:
        print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

    if ((epoch+1) % config["checkp_interval"] == 0) or ((epoch+1) == config['epochs']):
        file_save_model = f'models/{config["experiment_name"]}_{str(epoch+1).zfill(3)}.pth'
        save_model_and_results(
            mapping_net     = mapping_network,
            generator       = generator,
            discriminator   = discriminator,
            optimizer_m     = opt_mapping_network,
            optimizer_g     = opt_generator,
            optimizer_d     = opt_discriminator,
            results         = results,
            epoch           = epoch + 1,
            hyperparameters = config,
            file_name       = file_save_model,
        )

    if ((epoch+1) % config["sampling_interval"] == 0) or ((epoch+1) == config['epochs']):
        generate_grid_images(generator, 2, 4, epoch+1, config, device)

In [ ]:
# Mark the Weights & Bias run as finished
wandb.finish()

## Conclusion

In this article, we make a clean, simple, and readable implementation from scratch for a huge project which is StyleGAN2 using PyTorch. we try to replicate the original paper as closely as possible.

